# NeoN Python demo: vectors, interpolation, and Courant number

This second notebook demonstrates different Python capabilities of the installed `neon` package:

1. numpy interop with NeoN vectors,
2. token-driven interpolation in Python,
3. Courant number computation from mesh + face flux.


## 1) Imports and init

In [2]:
import numpy as np
import neon

if not globals().get("_neon_initialized", False):
    neon.initialize()
    _neon_initialized = True

exec = neon.SerialExecutor()
mesh = neon.create_1d_uniform_mesh(exec, 8)

print("executor:", exec.name())
print("cells:", mesh.n_cells())


executor: SerialExecutor
cells: 8


## 2) Capability: numpy view over NeoN vectors

`np.asarray(neon_vector)` gives a numpy view (CPU executors), which is handy for quick analysis and plotting.

In [4]:
v = neon.ScalarVector(exec, 10, 1.0)
a = np.asarray(v)

print("initial:", a)

a[3:7] = 4.5
print("after numpy edit:", a)
print("all values still 1.0?", neon.equal(v, 1.0))


initial: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
after numpy edit: [1.  1.  1.  4.5 4.5 4.5 4.5 1.  1.  1. ]
all values still 1.0? False


## 3) Capability: vector field interpolation selected by runtime tokens

We use `TokenList` to pick interpolation scheme at runtime.

In [5]:
U = neon.VectorVolumeField(exec, "U", mesh)
neon.fill(U.internal_vector(), neon.Vec3(1.0, 2.0, 3.0))

tokens = neon.TokenList()
tokens.insert_string("linear")

interp = neon.SurfaceInterpolationVec3(exec, mesh, tokens)
U_face = interp.interpolate(U)
U_face_np = np.asarray(U_face.internal_vector())

print("surface field shape:", U_face_np.shape)
print("first 3 face vectors:\n", U_face_np[:3])


surface field shape: (9, 3)
first 3 face vectors:
 [[1. 2. 3.]
 [1. 2. 3.]
 [1. 2. 3.]]
resetting face and cell centres


## 4) Capability: Courant number from mesh + face flux

`compute_co_num` returns `(max_Co, mean_Co)`.

In [6]:
face_count = mesh.n_internal_faces() + mesh.n_boundary_faces()
face_flux = neon.ScalarVector(exec, face_count)
neon.fill(face_flux, 1.0)

dt = 0.01
max_co, mean_co = neon.compute_co_num(mesh, face_flux, dt)

print("max Co:", max_co)
print("mean Co:", mean_co)


max Co: 0.08
mean Co: 0.08


## 5) Optional finalize

In [7]:
if globals().get("_neon_initialized", False):
    neon.finalize()
    _neon_initialized = False
    print("NeoN finalized")
else:
    print("NeoN already finalized")


NeoN finalized
Finalizing NeoN
